In [25]:
# 필요한 라이브러리 불러오기
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [26]:
# Titanic train.csv 파일 불러오기
df = pd.read_csv('/content/train.csv')

In [27]:
# 데이터 앞부분 확인
df.head()

df.info()

features = [
'Pclass',
'Sex',
'Age',
'SibSp',
'Parch',
'Fare',
'Embarked'
]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [28]:
X = df[features]

# 우리가 예측하려는 값
# Survived = 0 : 사망
# Survived = 1 : 생존
y = df['Survived']

In [29]:
numeric_features = [
'Pclass',
'Age',
'SibSp',
'Parch',
'Fare'
]

In [30]:
categorical_features = [
'Sex',
'Embarked'
]

In [31]:
# 숫자 데이터 전처리
numeric_transformer = Pipeline([
# 비어 있는 값은 중앙값으로 채움
('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())
])


In [32]:
# 문자 데이터 전처리
categorical_transformer = Pipeline([
# 비어 있는 값은 가장 많이 등장한 값으로 채움
('imputer', SimpleImputer(strategy='most_frequent')),
# male/female 같은 문자를 모델이 이해할 수 있도록 숫자로 변환
('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [33]:
preprocessor = ColumnTransformer([
('num', numeric_transformer, numeric_features),
('cat', categorical_transformer, categorical_features)
])

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2, # 전체 데이터의 20%를 평가용으로 사용
random_state=42, # 실행할 때마다 같은 결과가 나오도록 설정
stratify=y # 생존/사망 비율을 비슷하게 유지
)

In [35]:
model = Pipeline([
('preprocessor', preprocessor),
# 로지스틱 회귀 모델
('classifier', LogisticRegression(
C=1.0,
max_iter=1000
))
])

In [36]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Pclass', 'Age', 'SibSp',
                                                   'Parch', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sex', 'Embarked'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [37]:
y_pred = model.predict(X_test)

In [38]:
accuracy = accuracy_score(y_test, y_pred)

In [39]:
print("기본 모델 정확도:", accuracy)

기본 모델 정확도: 0.8044692737430168


In [40]:
param_grid = [
{
'classifier__C': [0.01, 0.1, 1, 10, 100],
'classifier__solver': ['liblinear'],
'classifier__penalty': ['l1', 'l2']
},
{
'classifier__C': [0.01, 0.1, 1, 10, 100],
'classifier__solver': ['lbfgs'],
'classifier__penalty': ['l2']
}
]

In [41]:
grid_search = GridSearchCV(
model,
param_grid,
cv=5, # 데이터를 5개로 나눠서 반복 평가
scoring='accuracy'
)

In [42]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Pclass',
                                                                          'Age',
                                                                          'SibSp',
                                                                          'Parch',
                                                                          'Fare']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('onehot',
                                                                                          OneHotEncoder(handle_unknown='ignore'))]),
                                                                         ['Sex',
                                                                          'Embarked'])])),
                                       ('classifier',
                                        LogisticRegression(max_iter=1000))]),
             param_grid=[{'classifier__C': [0.01, 0.1, 1, 10, 100],
                          'classifier__penalty': ['l1', 'l2'],
                          'classifier__solver': ['liblinear']},
                         {'classifier__C': [0.01, 0.1, 1, 10, 100],
                          'classifier__penalty': ['l2'],
                          'classifier__solver': ['lbfgs']}],
             scoring='accuracy')

In [43]:
print("가장 좋은 파라미터:")
print(grid_search.best_params_)

print("교차검증 최고 정확도:")
print(grid_search.best_score_)


가장 좋은 파라미터:
{'classifier__C': 0.1, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}
교차검증 최고 정확도:
0.8020388062641584


In [44]:
best_model = grid_search.best_estimator_

final_pred = best_model.predict(X_test)

final_accuracy = accuracy_score(y_test, final_pred)

print("최종 모델 정확도:", final_accuracy)

최종 모델 정확도: 0.7932960893854749


In [45]:
print(classification_report(y_test, final_pred))

              precision    recall  f1-score   support

           0       0.80      0.88      0.84       110
           1       0.78      0.65      0.71        69

    accuracy                           0.79       179
   macro avg       0.79      0.77      0.77       179
weighted avg       0.79      0.79      0.79       179

